# Indonesian Batik Pattern Classification with MobileNetV2

## Overview

**Batik** is a traditional Indonesian textile art involving intricate wax-resist dyeing techniques applied to fabric. In **2009**, UNESCO inscribed Indonesian Batik on the **Representative List of the Intangible Cultural Heritage of Humanity**, recognizing its deep cultural significance in ceremonies, rituals, and daily life across Java and beyond.

Each batik pattern carries its own philosophical meaning — from the royal **Parang** motif representing noble strength, to **Kawung** symbolizing purity, to **Mega Mendung** reflecting the calm of the Cirebon coastal culture.

### Why Classify Batik with AI?

- **Cultural preservation:** Automated classification helps archive and catalog thousands of regional patterns
- **Education:** Makes batik knowledge accessible to younger generations and researchers
- **Commerce:** Assists artisans and retailers in identifying and tagging inventory accurately
- **Heritage documentation:** Supports museums and cultural institutions in digitizing collections

### This Notebook

We build a **multi-class image classifier** using **Transfer Learning** with **MobileNetV2** pre-trained on ImageNet. The pipeline covers:

1. Dataset download and exploration
2. Data preprocessing and augmentation
3. Model architecture with frozen base
4. Training with callbacks
5. Evaluation: classification report + confusion matrix
6. Inference examples
7. Fine-tuning top layers
8. Model saving

> **Note:** This notebook requires Kaggle API credentials to download the dataset automatically. See the README for setup instructions (`~/.kaggle/kaggle.json`).

In [ ]:
import os
import json
import random
import zipfile
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# ─── Dataset Download ───────────────────────────────────────────────────────
# Requires Kaggle API credentials at ~/.kaggle/kaggle.json
# See README.md for setup instructions.
#
# Manual fallback: download from
#   https://www.kaggle.com/datasets/aldian/batik-kaggle
# and place images as: data/batik/<class_name>/<image>.jpg

DATASET = "aldian/batik-kaggle"
DATA_DIR = "data/batik"

if not os.path.exists(DATA_DIR):
    os.makedirs("data", exist_ok=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", DATASET, "-p", "data", "--unzip"],
        check=True
    )
    print("Dataset downloaded.")
else:
    print("Dataset already exists, skipping download.")

# Discover classes and count images
class_names = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

image_counts = {}
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_counts[cls] = len(images)

print(f"\nFound {len(class_names)} classes:")
for cls, count in image_counts.items():
    print(f"  {cls:25s} — {count:4d} images")

total = sum(image_counts.values())
print(f"\nTotal images: {total}")

## 1. Dataset Exploration

In [ ]:
# ─── 1a. Class Distribution Bar Chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab20.colors[:len(class_names)]
bars = ax.barh(list(image_counts.keys()), list(image_counts.values()), color=colors)
ax.set_xlabel("Number of Images", fontsize=12)
ax.set_title("Image Count per Batik Class", fontsize=14, fontweight='bold')
for bar, val in zip(bars, image_counts.values()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=10)
ax.set_xlim(0, max(image_counts.values()) * 1.15)
plt.tight_layout()
plt.show()
print(f"Average images per class: {total / len(class_names):.1f}")

# ─── 1b. Sample Images Grid (3 rows × 4 cols) ──────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle("Sample Batik Pattern Images", fontsize=16, fontweight='bold', y=1.01)
axes = axes.flatten()

sample_images = []
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        chosen = random.choice(imgs)
        sample_images.append((cls, os.path.join(cls_path, chosen)))

# Fill up to 12 samples
while len(sample_images) < 12 and len(sample_images) > 0:
    sample_images.append(random.choice(sample_images))
sample_images = sample_images[:12]

for ax, (cls, img_path) in zip(axes, sample_images):
    img = Image.open(img_path).convert('RGB').resize((224, 224))
    ax.imshow(img)
    ax.set_title(cls.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.axis('off')

for ax in axes[len(sample_images):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2. Data Preprocessing

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    subset="training",
    seed=SEED,
    shuffle=True
)

val_data = train_gen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    subset="validation",
    seed=SEED,
    shuffle=False
)

NUM_CLASSES = train_data.num_classes
CLASS_NAMES = list(train_data.class_indices.keys())

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Training batches  : {len(train_data)}")
print(f"Validation batches: {len(val_data)}")
print(f"Training samples  : {train_data.samples}")
print(f"Validation samples: {val_data.samples}")

# Visualize augmented samples
sample_batch, sample_labels = next(iter(train_data))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Augmented Training Samples", fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    ax.imshow(sample_batch[i])
    label_idx = np.argmax(sample_labels[i])
    ax.set_title(CLASS_NAMES[label_idx].replace('_', ' ').title(), fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Model Architecture (MobileNetV2 Transfer Learning)

We use **MobileNetV2** as the backbone, pre-trained on ImageNet with 1.4M parameters frozen. On top, we add:
- `GlobalAveragePooling2D` — reduces spatial dimensions
- `BatchNormalization` — stabilizes training
- `Dense(256, relu)` — task-specific feature learning
- `Dropout(0.5)` — prevents overfitting
- `Dense(NUM_CLASSES, softmax)` — output probability per class

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, optimizers

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # freeze base

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\nTotal parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters   : {total_params - trainable_params:,}")

## 4. Training

We train for up to 20 epochs with three callbacks:
- **EarlyStopping** (`patience=5`): stops when validation loss stops improving, restores best weights
- **ReduceLROnPlateau** (`patience=3`): halves the learning rate when progress stalls
- **ModelCheckpoint**: saves the best model to `models/batik_model.h5`

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

os.makedirs("models", exist_ok=True)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.5, verbose=1, min_lr=1e-7),
    ModelCheckpoint("models/batik_model.h5", save_best_only=True, verbose=1)
]

print("Starting training...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

best_val_acc = max(history.history['val_accuracy'])
best_epoch = history.history['val_accuracy'].index(best_val_acc) + 1
print(f"\nBest validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

In [ ]:
# ─── Training Curves ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History — Transfer Learning Phase", fontsize=14, fontweight='bold')

epochs_range = range(1, len(history.history['accuracy']) + 1)

ax1.plot(epochs_range, history.history['accuracy'], 'b-o', label='Train Accuracy', markersize=4)
ax1.plot(epochs_range, history.history['val_accuracy'], 'r-s', label='Val Accuracy', markersize=4)
ax1.set_title('Accuracy', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

ax2.plot(epochs_range, history.history['loss'], 'b-o', label='Train Loss', markersize=4)
ax2.plot(epochs_range, history.history['val_loss'], 'r-s', label='Val Loss', markersize=4)
ax2.set_title('Loss', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Evaluation

In [ ]:
# ─── Evaluate on validation set ─────────────────────────────────────────────
val_loss, val_acc = model.evaluate(val_data, verbose=0)
print(f"Validation Loss    : {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")

# Generate predictions
val_data.reset()
y_pred_probs = model.predict(val_data, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_data.classes

# Classification report
print("\n" + "="*60)
print("Classification Report")
print("="*60)
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print(report)

# Confusion matrix heatmap
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Confusion Matrix", fontsize=14, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
ax1.set_title('Counts', fontsize=12)
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.tick_params(axis='x', rotation=45)

sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
ax2.set_title('Normalized', fontsize=12)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Inference Example

In [ ]:
# ─── Inference on 5 random validation images ────────────────────────────────
val_data.reset()
val_filenames = val_data.filepaths
val_true_labels = val_data.classes

indices = random.sample(range(len(val_filenames)), min(5, len(val_filenames)))

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
fig.suptitle("Inference Examples — Predicted vs Actual", fontsize=14, fontweight='bold')

for ax, idx in zip(axes, indices):
    img_path = val_filenames[idx]
    true_label = CLASS_NAMES[val_true_labels[idx]]

    img = Image.open(img_path).convert('RGB').resize((224, 224))
    arr = np.array(img) / 255.0
    arr_batch = np.expand_dims(arr, 0)

    preds = model.predict(arr_batch, verbose=0)[0]
    pred_idx = np.argmax(preds)
    pred_label = CLASS_NAMES[pred_idx]
    confidence = preds[pred_idx] * 100

    correct = (pred_label == true_label)
    border_color = 'green' if correct else 'red'

    ax.imshow(img)
    ax.set_title(
        f"Pred: {pred_label.replace('_',' ').title()}\n"
        f"Conf: {confidence:.1f}%\n"
        f"True: {true_label.replace('_',' ').title()}",
        fontsize=8,
        color=border_color
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.show()

## 7. Fine-tuning (Unfreeze Top Layers)

Now we unfreeze the **top 30 layers** of the MobileNetV2 base and continue training with a much lower learning rate (`1e-5`). This allows the backbone to adapt its features to batik-specific patterns while retaining the low-level ImageNet representations in the earlier layers.

Key principles:
- Use a very small learning rate to avoid catastrophic forgetting
- Only unfreeze the top (last) layers — they capture the most domain-specific features
- Continue using EarlyStopping to prevent overfitting

In [ ]:
# Unfreeze top 30 layers for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_after = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable parameters after unfreezing: {trainable_after:,}")

model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.5, verbose=1, min_lr=1e-9),
    ModelCheckpoint("models/batik_model.h5", save_best_only=True, verbose=1)
]

print("\nStarting fine-tuning...")
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=fine_tune_callbacks,
    verbose=1
)

best_val_acc_fine = max(history_fine.history['val_accuracy'])
best_epoch_fine = history_fine.history['val_accuracy'].index(best_val_acc_fine) + 1
print(f"\nBest fine-tune validation accuracy: {best_val_acc_fine:.4f} at epoch {best_epoch_fine}")

In [ ]:
# ─── Re-evaluate after fine-tuning ──────────────────────────────────────────
val_loss_ft, val_acc_ft = model.evaluate(val_data, verbose=0)
print(f"Fine-tuned Validation Loss    : {val_loss_ft:.4f}")
print(f"Fine-tuned Validation Accuracy: {val_acc_ft:.4f} ({val_acc_ft*100:.2f}%)")

# Compare before vs after
print("\n" + "="*50)
print(f"{'Phase':<30} {'Val Accuracy':>15}")
print("-"*50)
print(f"{'Transfer Learning':<30} {val_acc*100:>14.2f}%")
print(f"{'After Fine-tuning':<30} {val_acc_ft*100:>14.2f}%")
delta = (val_acc_ft - val_acc) * 100
sign = '+' if delta >= 0 else ''
print(f"{'Improvement':<30} {sign}{delta:>13.2f}%")
print("="*50)

# Fine-tuning training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History — Fine-tuning Phase", fontsize=14, fontweight='bold')

ft_epochs = range(1, len(history_fine.history['accuracy']) + 1)

ax1.plot(ft_epochs, history_fine.history['accuracy'], 'b-o', label='Train Accuracy', markersize=4)
ax1.plot(ft_epochs, history_fine.history['val_accuracy'], 'r-s', label='Val Accuracy', markersize=4)
ax1.set_title('Accuracy (Fine-tuning)', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

ax2.plot(ft_epochs, history_fine.history['loss'], 'b-o', label='Train Loss', markersize=4)
ax2.plot(ft_epochs, history_fine.history['val_loss'], 'r-s', label='Val Loss', markersize=4)
ax2.set_title('Loss (Fine-tuning)', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Updated classification report after fine-tuning
val_data.reset()
y_pred_ft = np.argmax(model.predict(val_data, verbose=0), axis=1)
print("\nClassification Report (after fine-tuning):")
print(classification_report(val_data.classes, y_pred_ft, target_names=CLASS_NAMES))

## 8. Save Model & Class Names

In [ ]:
import json

os.makedirs("models", exist_ok=True)

model.save("models/batik_model.h5")

with open("models/class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)

print(f"Model saved to: models/batik_model.h5")
print(f"Class names saved to: models/class_names.json")
print(f"Classes: {CLASS_NAMES}")

model_size_mb = os.path.getsize("models/batik_model.h5") / (1024 * 1024)
print(f"Model file size: {model_size_mb:.1f} MB")

## 9. Conclusions

- **Transfer learning with MobileNetV2** proved highly effective for batik classification — pre-trained ImageNet weights capture low-level textures and patterns that generalize well to the intricate geometric structures found in batik fabric, giving strong baseline accuracy with minimal training time.

- **Fine-tuning the top 30 layers** with a low learning rate (`1e-5`) further improved validation accuracy by adapting higher-level feature maps to batik-specific visual patterns without losing the general representations from the frozen base.

- **Data augmentation** (rotation, flipping, shifting) was essential to prevent overfitting on the relatively small per-class image counts, making the model more robust to real-world image variations such as different lighting conditions, angles, and crop positions.

- **Potential applications** extend beyond classification: the trained embeddings (from the GlobalAveragePooling layer) could power image similarity search for batik archives, automated tagging in e-commerce platforms, and educational tools that help the public learn to recognize and appreciate Indonesia's diverse batik heritage.